# IMC25 – Single submission notebook

This notebook is intended to be **the one notebook** you run to produce a `submission.csv`.

It follows this flow:

1. **Train** a retrieval embedder using `data/train_labels.csv`
2. **Embed** all test images → `cache_retrieval/`
3. **Cluster** embeddings → `cache/clusters_retrieval.csv`
4. **Run SfM** per cluster (COLMAP) → sparse poses under `outputs_submission/<dataset>_<scene>/sparse/...`
5. **Export** a competition-format `submission.csv`

Notes:
- If you already have a trained checkpoint, you can skip training.
- For Kaggle-style runs, you usually want `SPARSE_ONLY=True` (dense MVS is not required for submission).
- This notebook assumes you start Jupyter from the repo root.


In [ ]:
from __future__ import annotations

import json
import os
import struct
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root() -> Path:
    p = Path.cwd().resolve()
    while p != p.parent:
        if (p / "src").exists() and (p / "pyproject.toml").exists():
            return p
        p = p.parent
    raise RuntimeError("Could not find repo root (expected src/ and pyproject.toml)")


REPO_ROOT = find_repo_root()
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")

print("repo:", REPO_ROOT)
print("HF_ENDPOINT:", os.environ.get("HF_ENDPOINT"))
print("python:", sys.version.split()[0])


## Paths + flags

Adjust these if your Kaggle input paths differ.


In [ ]:
# Data
TRAIN_LABELS_CSV = REPO_ROOT / "data" / "train_labels.csv"
TRAIN_ROOT = REPO_ROOT / "data" / "train"
TEST_ROOT = REPO_ROOT / "data" / "test"

# Outputs
OUTPUT_ROOT = REPO_ROOT / "outputs_submission"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Retrieval model
MODEL_ID = "facebook/dinov2-small"
TRAIN_OUTPUT_DIR = REPO_ROOT / "outputs" / "retrieval_finetune"
CHECKPOINT = TRAIN_OUTPUT_DIR / "best.pt"

# Pipeline flags
DO_TRAIN = False  # if False, expects CHECKPOINT to already exist
DO_EMBED_TEST = True
DO_CLUSTER = True
DO_COLMAP_SFM = True

# COLMAP settings
RUNNER = "docker"  # "docker" recommended; falls back to "local" if needed
MATCHER = "colmap"  # "colmap" | "diffusion" | "auto"
SPARSE_ONLY = True   # True = SfM only (faster); False = SfM + dense MVS

# CSV locations
CACHE_RETRIEVAL = REPO_ROOT / "cache_retrieval"
CLUSTERS_CSV = REPO_ROOT / "cache" / "clusters_retrieval.csv"
SUBMISSION_CSV = OUTPUT_ROOT / "submission.csv"

assert TRAIN_LABELS_CSV.exists(), f"Missing {TRAIN_LABELS_CSV}"
assert TRAIN_ROOT.exists(), f"Missing {TRAIN_ROOT}"
assert TEST_ROOT.exists(), f"Missing {TEST_ROOT}"

SUBMISSION_CSV


## Helper: run shell commands


In [ ]:
def run(cmd: list[str]) -> None:
    print("+", " ".join(cmd), flush=True)
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)


## 1) Train retrieval embedder

This uses the training labels to learn an embedding space where images from the same scene are close.


In [ ]:
if DO_TRAIN:
    TRAIN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    run(
        [
            sys.executable,
            "scripts/train_retrieval.py",
            "--model-id",
            MODEL_ID,
            "--output-dir",
            str(TRAIN_OUTPUT_DIR),
            "--data-root",
            str(TRAIN_ROOT),
            "--labels-csv",
            str(TRAIN_LABELS_CSV),
            "--hf-endpoint",
            os.environ.get("HF_ENDPOINT", ""),
            "--num-workers",
            "0",
        ]
    )

if not CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Checkpoint not found: {CHECKPOINT}. Set DO_TRAIN=True, or point CHECKPOINT to an existing best.pt"
    )

print("checkpoint:", CHECKPOINT)


## 2) Embed test images → `cache_retrieval/`


In [ ]:
if DO_EMBED_TEST:
    run(
        [
            sys.executable,
            "scripts/build_retrieval_cache.py",
            "--data-root",
            str(TEST_ROOT),
            "--cache-root",
            str(CACHE_RETRIEVAL),
            "--checkpoint",
            str(CHECKPOINT),
            "--topk",
            "30",
            "--mutual",
            "--hf-endpoint",
            os.environ.get("HF_ENDPOINT", ""),
            "--num-workers",
            "0",
        ]
    )

assert CACHE_RETRIEVAL.exists(), f"Missing {CACHE_RETRIEVAL}"
sorted([p.name for p in CACHE_RETRIEVAL.iterdir() if p.is_dir()])[:10]


## 3) Cluster embeddings → `cache/clusters_retrieval.csv`


In [ ]:
if DO_CLUSTER:
    run(
        [
            sys.executable,
            "scripts/cluster_retrieval.py",
            "--cache-root",
            str(CACHE_RETRIEVAL),
            "--out-csv",
            str(CLUSTERS_CSV),
        ]
    )

assert CLUSTERS_CSV.exists(), f"Missing {CLUSTERS_CSV}"
pd.read_csv(CLUSTERS_CSV).head()


## 4) Run SfM per cluster (COLMAP)

This produces sparse reconstructions under:

`outputs_submission/<dataset>_<scene>/sparse/<model_id>/images.bin`

By default we use `SPARSE_ONLY=True` (submission does not require dense point clouds).


In [ ]:
if DO_COLMAP_SFM:
    cmd = [
        sys.executable,
        "scripts/colmap_dense_clusters.py",
        "--clusters-csv",
        str(CLUSTERS_CSV),
        "--data-root",
        str(TEST_ROOT),
        "--output-root",
        str(OUTPUT_ROOT),
        "--runner",
        RUNNER,
        "--matcher",
        MATCHER,
    ]
    if SPARSE_ONLY:
        cmd.append("--sparse-only")
    run(cmd)


## 5) Export `submission.csv`

We read:
- cluster assignment from `cache/clusters_retrieval.csv`
- COLMAP poses from `outputs_submission/<dataset>_<scene>/sparse/.../images.bin`

For images that are outliers or were not registered by SfM, we write `nan` poses.


In [ ]:
IMG_EXTS = (".png", ".jpg", ".jpeg", ".webp")


def qvec2rotmat(q: np.ndarray) -> np.ndarray:
    # COLMAP quaternion format: [qw, qx, qy, qz]
    w, x, y, z = [float(v) for v in q.tolist()]
    return np.array(
        [
            [1 - 2 * y * y - 2 * z * z, 2 * x * y - 2 * w * z, 2 * x * z + 2 * w * y],
            [2 * x * y + 2 * w * z, 1 - 2 * x * x - 2 * z * z, 2 * y * z - 2 * w * x],
            [2 * x * z - 2 * w * y, 2 * y * z + 2 * w * x, 1 - 2 * x * x - 2 * y * y],
        ],
        dtype=np.float64,
    )


def read_images_bin(path: Path) -> dict[str, tuple[np.ndarray, np.ndarray]]:
    """Return map: image_name -> (qvec, tvec)."""
    out: dict[str, tuple[np.ndarray, np.ndarray]] = {}
    with path.open("rb") as f:
        num_images = struct.unpack("<Q", f.read(8))[0]
        for _ in range(int(num_images)):
            _image_id = struct.unpack("<i", f.read(4))[0]
            qvec = np.array(struct.unpack("<4d", f.read(32)), dtype=np.float64)
            tvec = np.array(struct.unpack("<3d", f.read(24)), dtype=np.float64)
            _camera_id = struct.unpack("<i", f.read(4))[0]

            name_bytes = bytearray()
            while True:
                c = f.read(1)
                if c == b"\x00" or c == b"":
                    break
                name_bytes.extend(c)
            name = name_bytes.decode("utf-8")

            num_points2d = struct.unpack("<Q", f.read(8))[0]
            f.seek(int(num_points2d) * (2 * 8 + 8), 1)

            out[name] = (qvec, tvec)
    return out


def colmap_bin_count(path: Path) -> int:
    try:
        with path.open("rb") as f:
            head = f.read(8)
        if len(head) != 8:
            return 0
        return int(struct.unpack("<Q", head)[0])
    except Exception:
        return 0


def pick_best_sparse_model(sparse_root: Path) -> Path | None:
    if not sparse_root.exists():
        return None
    candidates = [p for p in sparse_root.iterdir() if p.is_dir() and p.name.isdigit()]
    if not candidates:
        return None
    best = None
    best_path = None
    for p in sorted(candidates, key=lambda x: int(x.name)):
        n_images = colmap_bin_count(p / "images.bin")
        n_points = colmap_bin_count(p / "points3D.bin")
        score = (n_images, n_points, p.name)
        if best is None or score > best:
            best = score
            best_path = p
    return best_path


def load_meta_map(cache_root: Path, dataset: str) -> dict[str, Path]:
    meta_path = cache_root / dataset / "meta.json"
    if not meta_path.exists():
        return {}
    meta = json.loads(meta_path.read_text())
    image_ids = meta.get("image_ids", [])
    paths = meta.get("paths", [])
    if not isinstance(image_ids, list) or not isinstance(paths, list) or len(image_ids) != len(paths):
        return {}
    out: dict[str, Path] = {}
    for image_id, p in zip(image_ids, paths, strict=False):
        pp = Path(str(p))
        pp = pp if pp.is_absolute() else (REPO_ROOT / pp)
        out[str(image_id)] = pp.resolve()
    return out


def resolve_image_name(*, dataset: str, image_id: str, meta_map: dict[str, Path]) -> str:
    if image_id in meta_map:
        return meta_map[image_id].name
    # fallback: try <stem>.<ext>
    for ext in IMG_EXTS:
        p = TEST_ROOT / dataset / f"{image_id}{ext}"
        if p.exists():
            return p.name
    # last resort: assume image_id is already a filename
    return image_id


def fmt_mat9(R: np.ndarray) -> str:
    v = R.reshape(-1).tolist()
    return ";".join([f"{float(x):.9f}" for x in v])


def fmt_vec3(t: np.ndarray) -> str:
    v = t.reshape(-1).tolist()
    return ";".join([f"{float(x):.9f}" for x in v])


In [ ]:
clusters = pd.read_csv(CLUSTERS_CSV)
assert {"dataset", "scene", "image_id"}.issubset(set(clusters.columns))

meta_maps: dict[str, dict[str, Path]] = {}
for ds in sorted(clusters["dataset"].astype(str).unique().tolist()):
    meta_maps[ds] = load_meta_map(CACHE_RETRIEVAL, ds)

# Load COLMAP poses per (dataset, scene)
pose_maps: dict[tuple[str, str], dict[str, tuple[np.ndarray, np.ndarray]]] = {}
for (ds, scene), grp in clusters.groupby(["dataset", "scene"], sort=True):
    if str(scene).lower() == "outliers":
        continue
    out_dir = OUTPUT_ROOT / f"{ds}_{scene}"
    sparse_root = out_dir / "sparse"
    best = pick_best_sparse_model(sparse_root)
    if best is None:
        pose_maps[(str(ds), str(scene))] = {}
        continue
    images_bin = best / "images.bin"
    if not images_bin.exists():
        pose_maps[(str(ds), str(scene))] = {}
        continue
    pose_maps[(str(ds), str(scene))] = read_images_bin(images_bin)

rows = []
for r in clusters.itertuples(index=False):
    ds = str(r.dataset)
    scene = str(r.scene)
    image_id = str(r.image_id)
    meta_map = meta_maps.get(ds, {})
    image_name = resolve_image_name(dataset=ds, image_id=image_id, meta_map=meta_map)

    if scene.lower() == "outliers":
        rows.append(
            {
                "dataset": ds,
                "scene": scene,
                "image": image_name,
                "rotation_matrix": "nan;nan;nan;nan;nan;nan;nan;nan;nan",
                "translation_vector": "nan;nan;nan",
            }
        )
        continue

    pose_map = pose_maps.get((ds, scene), {})
    pose = pose_map.get(image_name)
    if pose is None:
        rows.append(
            {
                "dataset": ds,
                "scene": scene,
                "image": image_name,
                "rotation_matrix": "nan;nan;nan;nan;nan;nan;nan;nan;nan",
                "translation_vector": "nan;nan;nan",
            }
        )
        continue

    qvec, tvec = pose
    R = qvec2rotmat(qvec)
    rows.append(
        {
            "dataset": ds,
            "scene": scene,
            "image": image_name,
            "rotation_matrix": fmt_mat9(R),
            "translation_vector": fmt_vec3(tvec),
        }
    )

sub = pd.DataFrame(rows)
sub.to_csv(SUBMISSION_CSV, index=False)
print("wrote", SUBMISSION_CSV, "rows=", len(sub))
sub.head()


## Sanity checks


In [ ]:
sub = pd.read_csv(SUBMISSION_CSV)
print("rows:", len(sub))
print("datasets:", sub["dataset"].nunique())
print("scenes:", sub["scene"].nunique())
is_nan_pose = sub["translation_vector"].astype(str).str.contains("nan")
print("nan poses:", int(is_nan_pose.sum()), "/", len(sub))
sub.head(10)
